# Unsupervised Ellipsoid Fitting Algorithm

This notebook extends the hypersphere-based algorithm by replacing each spherical region with an ellipsoid. The motivation is that normal embeddings in DINOv2 feature space are unlikely to form locally isotropic clusters. Instead, neighbourhoods may stretch more strongly along some directions than others. Ellipsoids are therefore able to model local variance more naturally than hyperspheres.

The ellipsoid formulation has several advantages:

- Aligns each region with the natural variance structure of the local KNN neighbourhood.
- Reduces unused empty space compared with hyperspheres, since the boundary can contract along low-variance directions.
- It can reduce unnecessary overlap between neighbouring regions by following the dominant principal axes of the local embedding distribution.
- It is better suited to high-variance categories, where the normal embedding space may contain elongated or anisotropic regions.
- Provides additional interpretability through eigenvalues, eigenvectors, axis ratios, and local region structure.

Several changes were introduced compared with the hypersphere version:

- Growth is variance-scaled rather than uniform. Expansion along each axis is controlled by the relative eigenvalue contribution, so high-variance directions can grow more than low-variance directions.
- Candidate cleaning is weight-based. Instead of immediately removing a point when a candidate ellipsoid overlaps a previous region, the algorithm first reduces that point’s contribution to the ellipsoid  fit. If its weight reaches zero and overlap remains, the point is removed.
- Sparse ellipsoids require additional support. Unlike hyperspheres, ellipsoids fitted from very few points can become geometrically unstable. To address this, the covariance of a small candidate region is blended with covariance information from a previous ellipsoid.
- The current support strategy borrows covariance from the nearest ellipsoid by centre distance. This is a limitation, since the nearest ellipsoid may not be the most geometrically similar. Future work should select support using both spatial proximity and shape similarity.

In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime
from dataclasses import asdict

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH

EMBED_PATH = EMBEDS_DIR / "dino/20260801_221150"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "ellipsoid"
RESULTS_DIR = RESULTS / EMBED_NAME

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidFitter, EllipsoidCover, CandidateCleaner, EllipsoidEvaluator
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

fitter = EllipsoidFitter(support_points=5, reg=metadata.reg)
cleaner = CandidateCleaner(min_points=1, fitter=fitter)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)

eval = EllipsoidEvaluator()

In [6]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    collection = cover.run(
        embeds=cat_emb, output_dir=outputs_dir, 
        k_frac=metadata.K_frac, 
        start_growth=metadata.start_growth, min_growth=metadata.min_growth
        )
    
    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df, num_overlaps = eval.overlap(cat_emb, collection.ellipsoids)
    overlaps_df.to_csv(outputs_dir / f"overlaps.csv", index=False)

    good_any, good_counts = eval.inside_any_count(good_test_emb, collection.ellipsoids)
    defect_any, defect_counts = eval.inside_any_count(defect_test_emb, collection.ellipsoids)

    results_df, metrics = eval.evaluate_detection(good_test_emb, defect_test_emb, collection)
    results_df.to_csv(outputs_dir / f"results.csv", index=False)

    diagnostics = eval.bucket_diagnostics(results_df)

    aurocs[category] = metrics["auroc"]

    results = AlgorithmResults(
        config=metadata,
        category=category,
        n_shapes=len(collection),
        auroc=metrics["auroc"],
        normal_inside=int(good_any.sum()),
        defect_inside=int(defect_any.sum())
    )

    with open(outputs_dir / f"metadata.json", "w") as f:
        json.dump(asdict(results), f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])

Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [7]:
aurocs_df

,Category,AUROC
0,bottle,1.000000
1,cable,0.860570
2,capsule,0.875947
3,carpet,0.993981
4,grid,0.989140
5,hazelnut,0.990000
6,leather,1.000000
7,metal_nut,0.943304
8,pill,0.911620
9,screw,0.921500


In [8]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "ellipsoid_auroc_stats.csv", index=False)

aurocs_df = aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"ellipsoid_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.951,0.987,0.051,cable,0.861,bottle,1.0


In [9]:
display(diagnostics["n_points"])
display(diagnostics["eig_ratio"])
display(diagnostics["n_points_by_class"])
display(diagnostics["eig_ratio_aurocs"])

,mean,count
n_points_bucket,,
"(-0.001, 3.0]",0.948148,135
"(3.0, 5.0]",1.000000,11
"(5.0, 10.0]",1.000000,3
"(10.0, 100.0]",1.000000,2


,mean,count
eigval_ratio_bucket,,
"(6.17, 1.80793269582925e+16]",0.8500,40
"(1.80793269582925e+16, 3.3236190823926256e+16]",1.0000,59
"(3.3236190823926256e+16, 5.473479267990881e+16]",0.9375,16
"(5.473479267990881e+16, 8.803455828131274e+16]",1.0000,36


mean  count
n_points_bucket y_true                 
(-0.001, 3.0]   0.0     0.941176     17
                1.0     0.949153    118
(3.0, 5.0]      0.0     1.000000     10
                1.0     1.000000      1
(5.0, 10.0]     0.0     1.000000      3
                1.0          NaN      0
(10.0, 100.0]   0.0     1.000000      2
                1.0          NaN      0

,eigval_ratio_bucket,auroc,count
0,"(6.17, 1.80793269582925e+16]",0.931319,40
1,"(1.80793269582925e+16, 3.3236190823926256e+16]",1.000000,59
2,"(3.3236190823926256e+16, 5.473479267990881e+16]",0.933333,16
3,"(5.473479267990881e+16, 8.803455828131274e+16]",NaN,36


In [10]:
ellipsoids_df = collection.to_dataframe()

ellipsoids_df

,ellipsoid_id,raw_eig_ratio,reg_eig_ratio,pc95,rank,pc1_ratio,n_points,threshold,support_id,weights_mean,weights_min,n_reduced_weights
0,0,1.221356e+01,5075.746379,10,12,0.223561,13,11.069413,NaN,1.0,1.0,0
1,1,9.887881e+00,5713.330884,9,11,0.216631,12,10.078204,NaN,1.0,1.0,0
2,2,6.171233e+00,4880.093247,9,10,0.220614,11,9.086921,NaN,1.0,1.0,0
3,3,7.440695e+00,6621.786513,9,10,0.225590,11,9.087639,NaN,1.0,1.0,0
4,4,7.195737e+00,6471.369603,8,9,0.217988,10,8.097308,NaN,1.0,1.0,0
5,5,3.698121e+15,9023.433060,8,9,0.243185,10,8.098142,NaN,1.0,1.0,0
6,6,3.393948e+15,8977.407853,7,8,0.270492,9,7.109604,NaN,1.0,1.0,0
7,7,4.266271e+15,9880.665576,7,8,0.273234,9,7.109592,NaN,1.0,1.0,0
8,8,3.937357e+15,9008.969177,6,7,0.280589,8,6.123985,NaN,1.0,1.0,0
9,9,3.367220e+15,8418.103268,6,7,0.250816,8,6.123857,NaN,1.0,1.0,0


In [ ]:
print(ellipsoids_df["pc1_ratio"].mean())
print(ellipsoids_df["pc1_ratio"].median())
print(ellipsoids_df["pc95"].mean())
print(ellipsoids_df["pc95"].median())

0.4895988412464485
0.5364536629820947
4.066666666666666
<bound method Series.median of 0     10
1      9
2      9
3      9
4      8
5      8
6      7
7      7
8      6
9      6
10     6
11     6
12     6
13     5
14     5
15     5
16     5
17     4
18     4
19     4
20     3
21     3
22     3
23     3
24     3
25     2
26     2
27     2
28     2
29     2
30     2
31     2
32     2
33     2
34     2
35     2
36     2
37     2
38     2
39     2
40     2
41     2
42     2
43     2
44     1
Name: pc95, dtype: int64>
